In [1]:
from pathlib import Path
import os

from IPython.display import display
from pathlib import Path
from rasterio import features
import os
import pandas as pd
from satellites.shared.constants import SEED_NUMBER
from satellites.shared.notebook import configure_notebook
configure_notebook(seed=42, tensorflow=True)
import satellites.shared.io as io_l

from satellites.ml_classification import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

In [2]:
WORK_PATH = Path("C:/work_dir/volvi-2023-2024")

PARCELS_PATH = WORK_PATH / "1_planet_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
EDA_REPORT_PATH = WORK_PATH / "2_results_eda" / "eda_annotated_data.parquet"
OUTPUT_PATH = WORK_PATH / "3_ml_classification"

class_column = "label_gr"
parcel_id_column = "parcel_code"


In [3]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
eda_parcels = io_l.read_data(str(EDA_REPORT_PATH))
eda_parcels = eda_parcels[[parcel_id_column, "is_outlier_97_5pct"]]
gdf_parcels = gdf_parcels.merge(eda_parcels, on=parcel_id_column, how="left")
gdf_parcels.head(3)

Reading file: C:/work_dir/volvi-2023-2024/1_planet_parcel_stats/satellite_parcel_time_stats.geoparquet
Reading file: C:/work_dir/volvi-2023-2024/2_results_eda/eda_annotated_data.parquet


,parcel_code,initial_label_code,initial_label,predicted_label_code,predicted_label,prediction_probability,set_type,geometry,batch_number,intersected_pixel_count,meets_minimum_pixel_count,expected_pixel_count,temporal_filled_pixel_count,spatial_filled_pixel_count,temporal_filled_ratio,spatial_filled_ratio,data_reliability_score,pixel_area,geom_area,geom_interior_area_ratio,geom_compactness,geom_perimeter_area_ratio,geom_shape_index,geom_elongation,geom_shape_complexity_score,B1_coefficient_of_variation__20231201,B1_coefficient_of_variation__20240101,B1_coefficient_of_variation__20240201,B1_coefficient_of_variation__20240301,B1_coefficient_of_variation__20240401,B1_coefficient_of_variation__20240501,B1_coefficient_of_variation__20240601,B1_coefficient_of_variation__20240701,B1_coefficient_of_variation__20240801,B1_coefficient_of_variation__20240901,B1_max__20231201,B1_max__20240101,B1_max__20240201,B1_max__20240301,B1_max__20240401,B1_max__20240501,B1_max__20240601,B1_max__20240701,B1_max__20240801,B1_max__20240901,B1_mean__20231201,B1_mean__20240101,B1_mean__20240201,B1_mean__20240301,B1_mean__20240401,B1_mean__20240501,B1_mean__20240601,B1_mean__20240701,B1_mean__20240801,B1_mean__20240901,B1_median__20231201,B1_median__20240101,B1_median__20240201,B1_median__20240301,B1_median__20240401,B1_median__20240501,B1_median__20240601,B1_median__20240701,B1_median__20240801,B1_median__20240901,B1_min__20231201,B1_min__20240101,B1_min__20240201,B1_min__20240301,B1_min__20240401,B1_min__20240501,B1_min__20240601,B1_min__20240701,B1_min__20240801,B1_min__20240901,B1_range__20231201,B1_range__20240101,B1_range__20240201,B1_range__20240301,B1_range__20240401,B1_range__20240501,B1_range__20240601,B1_range__20240701,B1_range__20240801,B1_range__20240901,B1_sd__20231201,B1_sd__20240101,B1_sd__20240201,B1_sd__20240301,B1_sd__20240401,B1_sd__20240501,B1_sd__20240601,B1_sd__20240701,B1_sd__20240801,B1_sd__20240901,B1_variance__20231201,B1_variance__20240101,B1_variance__20240201,B1_variance__20240301,B1_variance__20240401,B1_variance__20240501,B1_variance__20240601,B1_variance__20240701,B1_variance__20240801,B1_variance__20240901,B2_coefficient_of_variation__20231201,B2_coefficient_of_variation__20240101,B2_coefficient_of_variation__20240201,B2_coefficient_of_variation__20240301,B2_coefficient_of_variation__20240401,B2_coefficient_of_variation__20240501,B2_coefficient_of_variation__20240601,B2_coefficient_of_variation__20240701,B2_coefficient_of_variation__20240801,B2_coefficient_of_variation__20240901,B2_max__20231201,B2_max__20240101,B2_max__20240201,B2_max__20240301,B2_max__20240401,B2_max__20240501,B2_max__20240601,B2_max__20240701,B2_max__20240801,B2_max__20240901,B2_mean__20231201,B2_mean__20240101,B2_mean__20240201,B2_mean__20240301,B2_mean__20240401,B2_mean__20240501,B2_mean__20240601,B2_mean__20240701,B2_mean__20240801,B2_mean__20240901,B2_median__20231201,B2_median__20240101,B2_median__20240201,B2_median__20240301,B2_median__20240401,B2_median__20240501,B2_median__20240601,B2_median__20240701,B2_median__20240801,B2_median__20240901,B2_min__20231201,B2_min__20240101,B2_min__20240201,B2_min__20240301,B2_min__20240401,B2_min__20240501,B2_min__20240601,B2_min__20240701,B2_min__20240801,B2_min__20240901,B2_range__20231201,B2_range__20240101,B2_range__20240201,B2_range__20240301,B2_range__20240401,B2_range__20240501,B2_range__20240601,B2_range__20240701,B2_range__20240801,B2_range__20240901,B2_sd__20231201,B2_sd__20240101,B2_sd__20240201,B2_sd__20240301,B2_sd__20240401,B2_sd__20240501,B2_sd__20240601,B2_sd__20240701,B2_sd__20240801,B2_sd__20240901,B2_variance__20231201,B2_variance__20240101,B2_variance__20240201,B2_variance__20240301,B2_variance__20240401,B2_variance__20240501,B2_variance__20240601,B2_variance__20240701,B2_variance__20240801,B2_variance__20240901,B3_coefficient_of_variation__20231201,B3_coefficient_of_variation__20240101,B3_coefficient_of_variation__20240201,B3_coefficient_of_variation__20240301,B3_coefficient_of_variation__

In [4]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 100].index

gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(gdf_parcels[class_column].value_counts())

label_gr
σιτάρι                       11140
χορτοδοτικές καλλιέργειες     6128
κριθάρι                       3911
αραβόσιτος                    2504
αγρανάπαυση                   1292
λοιπά σιτηρά                   962
ηλίανθος                       919
βαμβάκι                        531
ελαιώνες                       497
9999                           293
ελαιοκράμβη                    284
αρωματικά φυτά                 128
καρυδιές                       110
Name: count, dtype: int64


In [5]:
gdf_parcels = gdf_parcels.to_crs(2100)
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry"]
extra_features = ["elevation_mean_m", "GAIOIKANOT", "geom_interior_area_ratio", "data_reliability_score", "geom_shape_complexity_score", "is_outlier_97_5pct", "x", "y"]
temporal_features = ("_median", "_sd", "_range",)
selected_indices = ("R", "RVI", "NDMI", "NDRE", "SAVI", "NDVI", "EVI", "NDWI",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features) and any(idx in x for idx in selected_indices)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

Final features for modeling (158 features): ['EVI_median__20231201', 'EVI_median__20240101', 'EVI_median__20240201', 'EVI_median__20240301', 'EVI_median__20240401', 'EVI_median__20240501', 'EVI_median__20240601', 'EVI_median__20240701', 'EVI_median__20240801', 'EVI_median__20240901', 'EVI_range__20231201', 'EVI_range__20240101', 'EVI_range__20240201', 'EVI_range__20240301', 'EVI_range__20240401', 'EVI_range__20240501', 'EVI_range__20240601', 'EVI_range__20240701', 'EVI_range__20240801', 'EVI_range__20240901', 'EVI_sd__20231201', 'EVI_sd__20240101', 'EVI_sd__20240201', 'EVI_sd__20240301', 'EVI_sd__20240401', 'EVI_sd__20240501', 'EVI_sd__20240601', 'EVI_sd__20240701', 'EVI_sd__20240801', 'EVI_sd__20240901', 'NDRE_median__20231201', 'NDRE_median__20240101', 'NDRE_median__20240201', 'NDRE_median__20240301', 'NDRE_median__20240401', 'NDRE_median__20240501', 'NDRE_median__20240601', 'NDRE_median__20240701', 'NDRE_median__20240801', 'NDRE_median__20240901', 'NDRE_range__20231201', 'NDRE_range

,parcel_code,label_gr,geometry,EVI_median__20231201,EVI_median__20240101,EVI_median__20240201,EVI_median__20240301,EVI_median__20240401,EVI_median__20240501,EVI_median__20240601,EVI_median__20240701,EVI_median__20240801,EVI_median__20240901,EVI_range__20231201,EVI_range__20240101,EVI_range__20240201,EVI_range__20240301,EVI_range__20240401,EVI_range__20240501,EVI_range__20240601,EVI_range__20240701,EVI_range__20240801,EVI_range__20240901,EVI_sd__20231201,EVI_sd__20240101,EVI_sd__20240201,EVI_sd__20240301,EVI_sd__20240401,EVI_sd__20240501,EVI_sd__20240601,EVI_sd__20240701,EVI_sd__20240801,EVI_sd__20240901,NDRE_median__20231201,NDRE_median__20240101,NDRE_median__20240201,NDRE_median__20240301,NDRE_median__20240401,NDRE_median__20240501,NDRE_median__20240601,NDRE_median__20240701,NDRE_median__20240801,NDRE_median__20240901,NDRE_range__20231201,NDRE_range__20240101,NDRE_range__20240201,NDRE_range__20240301,NDRE_range__20240401,NDRE_range__20240501,NDRE_range__20240601,NDRE_range__20240701,NDRE_range__20240801,NDRE_range__20240901,NDRE_sd__20231201,NDRE_sd__20240101,NDRE_sd__20240201,NDRE_sd__20240301,NDRE_sd__20240401,NDRE_sd__20240501,NDRE_sd__20240601,NDRE_sd__20240701,NDRE_sd__20240801,NDRE_sd__20240901,NDVI_median__20231201,NDVI_median__20240101,NDVI_median__20240201,NDVI_median__20240301,NDVI_median__20240401,NDVI_median__20240501,NDVI_median__20240601,NDVI_median__20240701,NDVI_median__20240801,NDVI_median__20240901,NDVI_range__20231201,NDVI_range__20240101,NDVI_range__20240201,NDVI_range__20240301,NDVI_range__20240401,NDVI_range__20240501,NDVI_range__20240601,NDVI_range__20240701,NDVI_range__20240801,NDVI_range__20240901,NDVI_sd__20231201,NDVI_sd__20240101,NDVI_sd__20240201,NDVI_sd__20240301,NDVI_sd__20240401,NDVI_sd__20240501,NDVI_sd__20240601,NDVI_sd__20240701,NDVI_sd__20240801,NDVI_sd__20240901,NDWI_median__20231201,NDWI_median__20240101,NDWI_median__20240201,NDWI_median__20240301,NDWI_median__20240401,NDWI_median__20240501,NDWI_median__20240601,NDWI_median__20240701,NDWI_median__20240801,NDWI_median__20240901,NDWI_range__20231201,NDWI_range__20240101,NDWI_range__20240201,NDWI_range__20240301,NDWI_range__20240401,NDWI_range__20240501,NDWI_range__20240601,NDWI_range__20240701,NDWI_range__20240801,NDWI_range__20240901,NDWI_sd__20231201,NDWI_sd__20240101,NDWI_sd__20240201,NDWI_sd__20240301,NDWI_sd__20240401,NDWI_sd__20240501,NDWI_sd__20240601,NDWI_sd__20240701,NDWI_sd__20240801,NDWI_sd__20240901,SAVI_median__20231201,SAVI_median__20240101,SAVI_median__20240201,SAVI_median__20240301,SAVI_median__20240401,SAVI_median__20240501,SAVI_median__20240601,SAVI_median__20240701,SAVI_median__20240801,SAVI_median__20240901,SAVI_range__20231201,SAVI_range__20240101,SAVI_range__20240201,SAVI_range__20240301,SAVI_range__20240401,SAVI_range__20240501,SAVI_range__20240601,SAVI_range__20240701,SAVI_range__20240801,SAVI_range__20240901,SAVI_sd__20231201,SAVI_sd__20240101,SAVI_sd__20240201,SAVI_sd__20240301,SAVI_sd__20240401,SAVI_sd__20240501,SAVI_sd__20240601,SAVI_sd__20240701,SAVI_sd__20240801,SAVI_sd__20240901,elevation_mean_m,GAIOIKANOT,geom_interior_area_ratio,data_reliability_score,geom_shape_complexity_score,is_outlier_97_5pct,x,y
0,51480984,σιτάρι,MULTIPOLYGON (((...,0.1557922363,0.1539538950,0.1542798728,0.7935472727,0.7788815498,0.6031113863,0.2426075339,0.1977629364,0.2036465108,0.1401609331,0.0747739375,0.1354138255,0.1235613376,0.5109109879,0.3853182495,0.3208664358,0.1056070030,0.0961777717,0.0824667364,0.1226523295,0.0122888657,0.0193846412,0.0213744091,0.0932471623,0.0744708492,0.0554076913,0.0181581081,0.0162173658,0.0161089782,0.0191465322,0.1731718928,0.1322228312,0.1427661479,0.6411303282,0.6359639764,0.5475994349,0.2565837204,0.2241471261,0.2183940411,0.1502442062,0.0895435289,0.1404011399,0.1102058515,0.3554201424,0.2845575213,0.2799970210,0.1287728846,0.1137926280,0.0960345417,0.1312831938,0.0146073068,0.0212684712,0.0192627025,0.0609027665,0.0467830602,0.0482511400,0.0224314392,0.0206860634,0.0194399651,0.0220938930,0.26

In [6]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method=None,

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",

    random_state=SEED_NUMBER,
    # n_jobs=1,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=5,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=5,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        "bayesian",
        # "decision_tree",
        "extra_trees",
        "random_forest",
        #"gradient_boosting",
        #"lightgbm",
        "xgboost",
        "knn",
        "neural_network",
        # "tensorflow_neural_network",
        # "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

layer_features = max(16, min(len(features), 64))
n_temporal_channels = max(16, min(len(temporal_features), 64))
config.tune_params["tensorflow_neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["neural_network"]["model__hidden_layer_sizes"] = [(layer_features,), (layer_features, layer_features), (layer_features, layer_features // 2)]
config.tune_params["keras_lstm"]["model__lstm_units_1"] = [n_temporal_channels, 2 * n_temporal_channels]
pipeline = GeospatialClassificationPipeline(config)


In [7]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check


Function <run_check> started on 2026-09-11 21:02:04

Checking input data...

Resetting project output folder: C:\work_dir\volvi-2023-2024\3_ml_classification
Deleting folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\01_check'...
Successfully deleted folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\01_check' on attempt 1
Deleting folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\02_prepare'...
Successfully deleted folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\02_prepare' on attempt 1
Deleting folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\03_train'...
Successfully deleted folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\03_train' on attempt 1
Deleting folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\04_evaluate'...
Successfully deleted folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\04_evaluate' on attempt 1
Deleting folder 'C:\work_dir\volvi-2023-2024\3_ml_classification\05_predict'...
Successfully deleted folder 'C:\work_di

{'rows_total': 28699,
 'rows_labeled': 28699,
 'rows_unknown_target': 0,
 'n_features_requested': 158,
 'n_features_active': 158,
 'numeric_features': ['EVI_median__20231201',
  'EVI_median__20240101',
  'EVI_median__20240201',
  'EVI_median__20240301',
  'EVI_median__20240401',
  'EVI_median__20240501',
  'EVI_median__20240601',
  'EVI_median__20240701',
  'EVI_median__20240801',
  'EVI_median__20240901',
  'EVI_range__20231201',
  'EVI_range__20240101',
  'EVI_range__20240201',
  'EVI_range__20240301',
  'EVI_range__20240401',
  'EVI_range__20240501',
  'EVI_range__20240601',
  'EVI_range__20240701',
  'EVI_range__20240801',
  'EVI_range__20240901',
  'EVI_sd__20231201',
  'EVI_sd__20240101',
  'EVI_sd__20240201',
  'EVI_sd__20240301',
  'EVI_sd__20240401',
  'EVI_sd__20240501',
  'EVI_sd__20240601',
  'EVI_sd__20240701',
  'EVI_sd__20240801',
  'EVI_sd__20240901',
  'NDRE_median__20231201',
  'NDRE_median__20240101',
  'NDRE_median__20240201',
  'NDRE_median__20240301',
  'NDRE_medi

In [8]:
summary_prepare = pipeline.run_prepare()
summary_prepare


Function <run_prepare> started on 2026-09-11 21:02:16

Preparing data...
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/02_prepare/label_mapping.csv


E:\Eleni\git_repo\satellites\src\satellites\ml_classification\visuals\maps.py:156: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend()



Prepared train/test split: 22959 train rows, 5740 test rows.

Execution time of function <run_prepare> ended on 2026-09-11 21:02:27 - duration: 00:00:10


{'active_features': ['EVI_median__20231201',
  'EVI_median__20240101',
  'EVI_median__20240201',
  'EVI_median__20240301',
  'EVI_median__20240401',
  'EVI_median__20240501',
  'EVI_median__20240601',
  'EVI_median__20240701',
  'EVI_median__20240801',
  'EVI_median__20240901',
  'EVI_range__20231201',
  'EVI_range__20240101',
  'EVI_range__20240201',
  'EVI_range__20240301',
  'EVI_range__20240401',
  'EVI_range__20240501',
  'EVI_range__20240601',
  'EVI_range__20240701',
  'EVI_range__20240801',
  'EVI_range__20240901',
  'EVI_sd__20231201',
  'EVI_sd__20240101',
  'EVI_sd__20240201',
  'EVI_sd__20240301',
  'EVI_sd__20240401',
  'EVI_sd__20240501',
  'EVI_sd__20240601',
  'EVI_sd__20240701',
  'EVI_sd__20240801',
  'EVI_sd__20240901',
  'NDRE_median__20231201',
  'NDRE_median__20240101',
  'NDRE_median__20240201',
  'NDRE_median__20240301',
  'NDRE_median__20240401',
  'NDRE_median__20240501',
  'NDRE_median__20240601',
  'NDRE_median__20240701',
  'NDRE_median__20240801',
  'NDRE_

In [9]:
summary_train = pipeline.run_train()
summary_train


Function <run_train> started on 2026-09-11 21:02:27

Training models...
Reading file: C:/work_dir/volvi-2023-2024/3_ml_classification/02_prepare/label_mapping.csv
____________________________________________________________________________________________________

Training model random_forest
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/03_train/models/random_forest/cv_results.csv

Saved artifacts for random_forest: model=C:\work_dir\volvi-2023-2024\3_ml_classification\03_train\models\random_forest\best_model.joblib, cv_results=C:\work_dir\volvi-2023-2024\3_ml_classification\03_train\models\random_forest\cv_results.csv, plot=C:\work_dir\volvi-2023-2024\3_ml_classification\03_train\models\random_forest\search_results.png

Model random_forest finished in duration: 00:03:52
____________________________________________________________________________________________________

Training model extra_trees
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/03_train/

{'models_trained': ['xgboost',
  'random_forest',
  'extra_trees',
  'bagging',
  'neural_network',
  'knn',
  'bayesian'],
 'models_failed': [],
 'best_cv_model': 'xgboost',
 'cv_ranking_method': 'score_minus_std'}

In [10]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate


Function <run_evaluate> started on 2026-09-11 21:38:04

Evaluating models...
Reading file: C:/work_dir/volvi-2023-2024/3_ml_classification/02_prepare/label_mapping.csv
Reading file: C:/work_dir/volvi-2023-2024/3_ml_classification/03_train/training_summary.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/random_forest_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/extra_trees_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/knn_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/neural_network_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/bagging_classification_report.csv


c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/bayesian_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/reports/xgboost_classification_report.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/model_metrics_train.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/model_metrics_test.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/interpretability/xgboost_feature_importance.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/interpretability/random_forest_feature_importance.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/interpretability/extra_trees_feature_importance.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/interpretability/interpretability_summary.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/04_evaluate/mo

{'selection_type': 'soft_voting',
 'selected_models': ['xgboost',
  'random_forest',
  'extra_trees',
  'bagging',
  'neural_network'],
 'selected_metric': 'cv_score_minus_std',
 'selected_score': 0.328327143679316,
 'selection_source': 'cross_validation',
 'labels': ['9999',
  'αγρανάπαυση',
  'αραβόσιτος',
  'αρωματικά φυτά',
  'βαμβάκι',
  'ελαιοκράμβη',
  'ελαιώνες',
  'ηλίανθος',
  'καρυδιές',
  'κριθάρι',
  'λοιπά σιτηρά',
  'σιτάρι',
  'χορτοδοτικές καλλιέργειες'],
 'class_probability_multipliers': {'9999': 2.0,
  'αγρανάπαυση': 1.5,
  'αραβόσιτος': 0.8,
  'αρωματικά φυτά': 0.8,
  'βαμβάκι': 0.8,
  'ελαιοκράμβη': 0.8,
  'ελαιώνες': 0.8,
  'ηλίανθος': 0.8,
  'καρυδιές': 1.5,
  'κριθάρι': 1.0,
  'λοιπά σιτηρά': 1.0,
  'σιτάρι': 1.2,
  'χορτοδοτικές καλλιέργειες': 1.5},
 'probability_optimization': {'baseline_oof_macro_f1': 0.4654755516919391,
  'optimized_oof_macro_f1': 0.4925707186959241,
  'baseline_oof_accuracy': 0.5600853695718455,
  'optimized_oof_accuracy': 0.576549501284899

In [11]:
summary_predict = pipeline.run_predict()
summary_predict


Function <run_predict> started on 2026-09-11 21:46:31

Classify all data...
Reading file: C:/work_dir/volvi-2023-2024/3_ml_classification/02_prepare/label_mapping.csv
Writing file: C:/work_dir/volvi-2023-2024/3_ml_classification/05_predict/final_predictions_preview.csv

Filled 0 unknown labels.

Execution time of function <run_predict> ended on 2026-09-11 22:03:33 - duration: 00:17:02


{'selection_type': 'soft_voting',
 'selected_models': ['xgboost',
  'random_forest',
  'extra_trees',
  'bagging',
  'neural_network'],
 'rows_filled': 0,
 'rows_unknown_original': 0,
 'rows_needing_review': 16615,
 'prediction_max_probability_column': 'prediction_max_probability',
 'review_decision_basis': 'rank_confidence',
 'rank_confidence_enabled': True,
 'rank_confidence_contract_source': 'selection_summary',
 'confidence_method': 'rank_consensus_with_class_reliability_guard',
 'class_reliability_method': 'oof_precision',
 'rank_confidence_minimum_models': 3,
 'rank_confidence_level_counts': {'LOW': 16615, 'MEDIUM': 9063, 'HIGH': 3021},
 'output_path': 'C:\\work_dir\\volvi-2023-2024\\3_ml_classification\\05_predict\\final_predictions.joblib',
 'inspection_scoring_enabled': True,
 'inspection_scoring_available': True,
 'inspection_weights': {'model': 0.6, 'data': 0.25, 'geometry': 0.15},
 'inspection_need_counts': {'MEDIUM': 19323,
  'MEDIUM_HIGH': 7839,
  'LOW': 1241,
  'HIGH': 1

In [12]:
# Show only rows that were unknown during training
predictions_gdf = pd.read_csv(os.path.join(str(OUTPUT_PATH), "05_predict", "final_predictions_preview.csv"))
predictions_gdf[parcel_id_column] = predictions_gdf[parcel_id_column].astype(str)
print(len(predictions_gdf), "rows in predictions_gdf")
display(predictions_gdf.head())

28699 rows in predictions_gdf


,parcel_code,label_gr,label_gr_prediction,predicted_class,prediction_max_probability,prediction_needs_review,label_gr_filled,prediction_confidence_level,prediction_confidence_valid,prediction_confidence_reason,prediction_mean_borda,prediction_rank_range,prediction_mean_rank,prediction_median_rank,prediction_rank_std,prediction_rank_iqr,prediction_borda_winner,prediction_borda_winner_tied,prediction_rank_agrees_with_final,prediction_rank_confidence_level,prediction_class_oof_precision,prediction_class_oof_support,prediction_class_reliability_level,prediction_class_reliability_valid,prediction_class_reliability_reason,prediction_borda_margin,prediction_top1_agreement,prediction_top2_agreement,prediction_top3_agreement,prediction_runner_up_class,prediction_runner_up_borda,prediction_rank_models_used,prediction_rank_confidence_valid,prediction_rank_confidence_reason,data_reliability_score,geom_shape_complexity_score,label_prediction_status,confidence_risk,data_risk,geometry_risk,inspection_score,inspection_need_base,inspection_need,inspection_check_type,inspection_reasons
0,51480984,σιτάρι,χορτοδοτικές καλ...,χορτοδοτικές καλ...,0.3167768306,True,σιτάρι,LOW,True,soft_vote_borda_...,86.6666666667,3.0000000000,2.6000000000,3.0000000000,1.0198039027,1.0000000000,σιτάρι,False,False,LOW,0.5770617395,4438,LOW,True,precision_below_...,-11.6666666667,0.2000000000,0.4000000000,0.8000000000,σιτάρι,98.3333333333,5,True,soft_vote_borda_...,1.0000000000,0.4083000400,DIFFERENT,1.0000000000,0.0000000000,0.6236454232,69.3500000000,HIGH,MEDIUM_HIGH,INSUFFICIENT_EO_...,DECLARATION_PRED...
1,237535570,σιτάρι,σιτάρι,σιτάρι,0.5745637316,False,σιτάρι,MEDIUM,True,class_reliabilit...,100.0000000000,0.0000000000,1.0000000000,1.0000000000,0.0000000000,0.0000000000,σιτάρι,False,True,HIGH,0.7076276425,8089,MEDIUM,True,precision_below_...,10.0000000000,1.0000000000,1.0000000000,1.0000000000,κριθάρι,90.0000000000,5,True,rank_confidence_...,0.9999784000,0.4688159500,SAME,0.5000000000,0.0000215769,0.7282135266,40.9200000000,MEDIUM,MEDIUM,INSUFFICIENT_EO_...,INSUFFICIENT_EO_...
2,237535579,σιτάρι,σιτάρι,σιτάρι,0.3470814109,False,σιτάρι,MEDIUM,True,class_reliabilit...,98.3333333333,1.0000000000,1.2000000000,1.0000000000,0.4000000000,0.0000000000,σιτάρι,False,True,HIGH,0.7076276425,8089,MEDIUM,True,precision_below_...,5.0000000000,0.8000000000,1.0000000000,1.0000000000,κριθάρι,93.3333333333,5,True,rank_confidence_...,0.9997611000,0.2529787000,SAME,0.5000000000,0.0002388954,0.3039827172,34.5700000000,MEDIUM,MEDIUM,INSUFFICIENT_EO_...,INSUFFICIENT_EO_...
3,237535543,χορτοδοτικές καλ...,αγρανάπαυση,αγρανάπαυση,0.2368333889,True,χορτοδοτικές καλ...,LOW,True,class_reliabilit...,93.3333333333,3.0000000000,1.8000000000,1.0000000000,1.1661903790,1.0000000000,αγρανάπαυση,False,True,MEDIUM,0.2212457012,2617,LOW,True,precision_below_...,8.3333333333,0.6000000000,0.8000000000,0.8000000000,αρωματικά φυτά,85.0000000000,5,True,rank_confidence_...,1.0000000000,0.4132014800,DIFFERENT,1.0000000000,0.0000000000,0.6323913725,69.4900000000,HIGH,MEDIUM_HIGH,INSUFFICIENT_EO_...,DECLARATION_PRED...
4,237535582,σιτάρι,σιτάρι,σιτάρι,0.3878488187,False,σιτάρι,MEDIUM,True,class_reliabilit...,100.0000000000,0.0000000000,1.0000000000,1.0000000000,0.0000000000,0.0000000000,σιτάρι,False,True,HIGH,0.7076276425,8089,MEDIUM,True,precision_below_...,8.3333333333,1.0000000000,1.0000000000,1.0000000000,κριθάρι,91.6666666667,5,True,rank_confidence_...,1.0000000000,0.3146006200,SAME,0.5000000000,0.0000000000,0.4361824454,36.5400000000,MEDIUM,MEDIUM,INSUFFICIENT_EO_...,INSUFFICIENT_EO_...


In [13]:
# From test results
test_results = pd.read_csv(os.path.join(str(OUTPUT_PATH), "04_evaluate", "confidence", "rank_confidence_test.csv"))
print(len(test_results), "rows in test_results")
test_results["accuracy"] = test_results["true_label"] == test_results["predicted_class"]
accuracy = test_results["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")

# From final predictions
test_results[parcel_id_column] = test_results[parcel_id_column].astype(str)
test_ids =list(test_results[parcel_id_column].unique())
classified_gdf = predictions_gdf[predictions_gdf[parcel_id_column].isin(test_ids)].copy()
print(len(classified_gdf), "rows in classified_gdf after filtering with test_ids")
classified_gdf["accuracy"] = classified_gdf[class_column] == classified_gdf[f"{class_column}_prediction"]
accuracy = classified_gdf["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")

5740 rows in test_results
Accuracy on data: 58.78%
5740 rows in classified_gdf after filtering with test_ids
Accuracy on data: 71.71%
